# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
# TODO: Import the necessary libs
# For example: 
# import os

# from lib.agents import Agent
# from lib.llm import LLM
# from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
# from lib.tooling import tool
__import__("pysqlite3")
import sys
sys.modules["sqlite3"] = sys.modules.pop("pysqlite3")


import os
import chromadb
from chromadb.utils import embedding_functions
from chromadb.api.models.Collection import Collection

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

In [3]:
# TODO: Load environment variables
from dotenv import load_dotenv
load_dotenv("config.env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "https://openai.vocareum.com/v1")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

if not OPENAI_API_KEY or not TAVILY_API_KEY:
    raise ValueError("Missing OPENAI_API_KEY or TAVILY_API_KEY — check config.env")

print("OPENAI_API_KEY loaded:", bool(OPENAI_API_KEY))

OPENAI_API_KEY loaded: True


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [4]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
# chroma_client = chromadb.PersistentClient(path="chromadb")
# collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game

chroma_client = chromadb.PersistentClient(path="./chromadb")

embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY,
    api_base=OPENAI_BASE_URL,
)
collection = chroma_client.get_collection(name="udaplay", embedding_function=embedding_fn)

@tool
def retrieve_game(query: str, n_results: int = 3) -> list[dict]:
    """
    Semantic search over the internal video game vector database.

    Args:
        query: a natural language question about a game (title, platform,
            release year, genre, publisher, or description).
        n_results: how many candidate games to return.

    Returns:
        A list of dicts with Name, Platform, YearOfRelease, Genre, Publisher,
        Description, and a "distance" score (lower = more relevant).
    """
    results = collection.query(query_texts=[query], n_results=n_results)
    metadatas = results.get("metadatas", [[]])[0]
    distances = results.get("distances", [[]])[0]

    return [
        {
            "Name": m.get("Name"),
            "Platform": m.get("Platform"),
            "YearOfRelease": m.get("YearOfRelease"),
            "Genre": m.get("Genre"),
            "Publisher": m.get("Publisher"),
            "Description": m.get("Description"),
            "distance": dist,
        }
        for m, dist in zip(metadatas, distances)
    ]

#### Evaluate Retrieval Tool

In [5]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result

#chroma_client = chromadb.PersistentClient(path="chromadb")
#collection = chroma_client.get_collection("udaplay")

from pydantic import BaseModel, Field

class EvaluationReport(BaseModel):
    useful: bool = Field(description="Whether the retrieved documents are sufficient to answer the question")
    description: str = Field(description="Detailed explanation of the verdict, including what's missing if not useful")

judge_llm = LLM(model="gpt-4o-mini")  # check lib/llm.py if this constructor differs

import json
@tool
def evaluate_retrieval(question: str, retrieved_docs: list[dict]) -> EvaluationReport:
    """
    Judges whether the documents retrieved by retrieve_game are enough to
    confidently answer the user's question.

    Args:
        question: the user's original question.
        retrieved_docs: the list of documents returned by retrieve_game.

    Returns:
        An EvaluationReport with `useful` (bool) and `description` (str).
    """
    docs_text = "\n".join(
        f"- {d.get('Name')} ({d.get('Platform')}, {d.get('YearOfRelease')}), "
        f"Publisher: {d.get('Publisher')} — {d.get('Description')}"
        for d in retrieved_docs
    ) or "No documents were retrieved."

    messages = [
        SystemMessage(content=(
            "Your task is to evaluate if the documents are enough to respond to the query. "
            "Give a detailed explanation, so it's possible to take an action to accept it or not. "
            "Be strict: if the documents don't directly and specifically answer the question, "
            "mark useful as false."
        )),
        UserMessage(content=f"Question: {question}\n\nRetrieved documents:\n{docs_text}"),
    ]

    response = judge_llm.invoke(messages, response_format=EvaluationReport)
    result = json.loads(response.content)
    return EvaluationReport(**result)


In [6]:
# Diagnostic cell — run this by itself
test_messages = [
    SystemMessage(content=(
        "Your task is to evaluate if the documents are enough to respond to the query. "
        "Give a detailed explanation, so it's possible to take an action to accept it or not."
    )),
    UserMessage(content=(
        "Question: When was Pokémon Gold and Silver released?\n\n"
        "Retrieved documents:\n"
        "- Pokémon Gold and Silver (Game Boy Color, 1999), Publisher: Nintendo — "
        "Second generation Pokémon games."
    )),
]

test_response = judge_llm.invoke(test_messages, response_format=EvaluationReport)
print(type(test_response))
print(test_response)

<class 'lib.messages.AIMessage'>
role='assistant' content='{"useful":true,"description":"The retrieved document provides the release year of Pokémon Gold and Silver, stating that they were released in 1999. This directly answers the query regarding the release date of the games. No additional information is needed to respond to the question."}' tool_calls=None token_usage=TokenUsage(prompt_tokens=156, completion_tokens=56, total_tokens=212)


#### Game Web Search Tool

In [7]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 

from tavily import TavilyClient

tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

@tool
def game_web_search(question: str) -> str:
    """
    Falls back to a web search when the internal knowledge base doesn't have
    enough information to answer the question.

    Args:
        question: a question about the game industry (release info, platforms,
            publishers, or what a studio is currently working on).

    Returns:
        A short summary string built from the top web results, each tagged
        with its source URL.
    """
    response = tavily_client.search(query=question, max_results=5, include_answer=True)

    parts = []
    if response.get("answer"):
        parts.append(f"Summary: {response['answer']}")
    for r in response.get("results", []):
        parts.append(f"- {r['title']}: {r['content'][:300]}... (Source: {r['url']})")

    return "\n".join(parts) if parts else "No web results found."

In [8]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed

instructions = """
You are UdaPlay, a video game research assistant.

For every question about video games (titles, release dates, platforms,
genres, publishers, or what a studio is currently working on):

1. Call retrieve_game to search the internal knowledge base first.
2. Call evaluate_retrieval to judge whether those documents are enough to
   answer confidently.
3. If evaluate_retrieval says the documents are NOT useful (or nothing was
   retrieved), call game_web_search for current information.
4. Write a final answer that:
   - Directly answers the question
   - Cites the source: "(Source: internal database)" or
     "(Source: web search — <url>)"
   - States a confidence level (High/Medium/Low)

Never guess. If neither source has the answer, say so plainly.
"""

udaplay_agent = Agent(
    model_name="gpt-4o-mini",
    instructions=instructions,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
)

In [9]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?

session_id = "udaplay-demo-session"

def ask(question: str):
    print(f"Q: {question}\n")

    run = udaplay_agent.invoke(question, session_id=session_id)
    final_state = run.get_final_state()
    conversation = final_state["messages"]

    tools_used = []
    for msg in conversation:
        tool_calls = getattr(msg, "tool_calls", None)
        if tool_calls:
            for tc in tool_calls:
                name = tc.get("name") if isinstance(tc, dict) else getattr(tc, "name", None)
                if name:
                    tools_used.append(name)

    final_answer = None
    for msg in reversed(conversation):
        role = getattr(msg, "role", None)
        content = getattr(msg, "content", None)
        if role == "assistant" and content:
            final_answer = content
            break

    print(f"Tools used: {tools_used if tools_used else 'none'}\n")
    print(f"Final answer:\n{final_answer}")
    print("\n" + "=" * 80 + "\n")

    return run, tools_used, final_answer


ask("When was Pokémon Gold and Silver released?")
ask("Which was the first 3D platformer Mario game?")
ask("Was Mortal Kombat X released for PlayStation 5?")
ask("What is Rockstar Games working on right now?")

Q: When was Pokémon Gold and Silver released?

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Tools used: none

Final answer:
Pokémon Gold and Silver were released in Japan on November 21, 1999, and in North America on October 15, 2000. (Source: web search — https://www.pokemon.com/us/pokemon-video-games/pokemon-gold-version-and-pokemon-silver-version) 

Confidence level: High


Q: Which was the first 3D platformer Mario game?

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMa

(Run('bba3f61b-b57b-4bdb-b893-7e978837049b'),
 [],
 'Rockstar Games is currently developing new content for **Grand Theft Auto Online** and **Red Dead Online**, along with recent updates and events for these games. (Source: web search — https://downdetector.com/status/rockstar-games)\n\nConfidence level: High')

### (Optional) Advanced

In [63]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes